# 05.1 Integers

Python's integers have a property almost no other mainstream language shares:
they have **no maximum value**. There is no `int32`, no `long`, no overflow. An
integer grows until you run out of memory.

That design choice has consequences worth understanding.

## Theory

### Arbitrary precision

In C or Java, an `int` is a fixed number of bits — typically 32 or 64. Exceed the
maximum and the value **wraps around** silently:

```c
int x = 2147483647;   // the largest 32-bit signed int
x = x + 1;            // becomes -2147483648  (a real bug)
```

Python has no such limit. It stores integers as a variable-length sequence of
digits internally, allocating more space as needed.

```python
2 ** 1000    # a 302-digit number, computed exactly
```

### How CPython stores them

An integer is an object with a header plus an array of 30-bit "digits". This is
why `sys.getsizeof()` grows with the magnitude of the number — a bigger integer
genuinely occupies more memory.

The consequences:

- **No overflow bugs** — ever
- **Exact arithmetic** at any size
- **Slower** than machine integers, and slower still as numbers grow
- Memory use scales with magnitude

### Integer literals

Python accepts four bases, plus underscores for readability:

| Form | Prefix | Example |
|---|---|---|
| Decimal | none | `255` |
| Binary | `0b` | `0b11111111` |
| Octal | `0o` | `0o377` |
| Hexadecimal | `0x` | `0xFF` |

Underscores are ignored entirely: `1_000_000` is exactly `1000000`.

### The two division operators

This is the part beginners get wrong most often:

- `/` is **true division** and **always** returns a `float`
- `//` is **floor division** and rounds **towards negative infinity**

`10 / 5` is `2.0`, not `2`. And `-7 // 2` is `-4`, not `-3`.

In [ ]:
import sys

# Integers have no maximum. Watch them grow.
powers = [10, 50, 100, 500]

print("Value                                   Digits   Bytes")
print("-" * 62)

for exponent in powers:
    value = 2 ** exponent

    # Show a preview of very large numbers rather than the whole thing.
    text = str(value)
    preview = text if len(text) <= 30 else text[:27] + "..."

    print(preview.ljust(40), str(len(text)).ljust(8), sys.getsizeof(value))

print("")
print("No overflow, no wrapping. The number simply uses more memory.")

# A genuinely enormous value, computed exactly.
enormous = 2 ** 10000
print("")
print("2 ** 10000 has", len(str(enormous)), "digits and is exact.")

In [ ]:
import sys

# Compare with what a fixed-width language would do.
int32_max = 2 ** 31 - 1

print("The largest 32-bit signed integer:", int32_max)
print("In C, adding 1 would WRAP to:      ", -(2 ** 31))
print("In Python, adding 1 gives:         ", int32_max + 1)

print("")
print("This class of bug simply does not exist in Python.")

# Factorials illustrate the growth well.
import math

print("")
print("Factorials stay exact at any size:")
for number in [10, 20, 50]:
    result = math.factorial(number)
    text = str(result)
    preview = text if len(text) <= 40 else text[:37] + "..."
    print(f"   {number}! = {preview}  ({len(text)} digits)")

print("")
print("20! already exceeds a 64-bit integer. Python does not care.")

## Integer literals in four bases

All four produce the same kind of object — the prefix only affects how you write
it, not what is stored.

In [ ]:
# The same value, written four ways.
as_decimal = 255
as_binary = 0b11111111
as_octal = 0o377
as_hexadecimal = 0xFF

print("Four ways to write 255:")
print("   decimal 255       ->", as_decimal)
print("   binary  0b11111111 ->", as_binary)
print("   octal   0o377     ->", as_octal)
print("   hex     0xFF      ->", as_hexadecimal)
print("")
print("All equal?", as_decimal == as_binary == as_octal == as_hexadecimal)
print("All the same type?", type(as_decimal).__name__)

# Converting back to a string in each base.
value = 255
print("")
print("Converting 255 to each base:")
print("   bin(255) ->", bin(value))
print("   oct(255) ->", oct(value))
print("   hex(255) ->", hex(value))

# format() gives the digits without the prefix.
print("")
print("Without the prefix, padded to 8 digits:")
print("   format(255, '08b') ->", format(value, "08b"))
print("   format(255, '04x') ->", format(value, "04x"))

In [ ]:
# int() parses a string in any base you specify.
print("Parsing strings into integers:")
print("   int('255')          ->", int("255"))
print("   int('11111111', 2)  ->", int("11111111", 2))
print("   int('377', 8)       ->", int("377", 8))
print("   int('FF', 16)       ->", int("FF", 16))
print("   int('0xFF', 16)     ->", int("0xFF", 16), "<- prefix allowed")

# Base 0 means "work it out from the prefix".
print("   int('0b1010', 0)    ->", int("0b1010", 0))

# Underscores make long literals readable and are ignored by Python.
population = 1_400_000_000
card_number = 1234_5678_9012_3456

print("")
print("Underscores are purely visual:")
print("   1_400_000_000 ->", population)
print("   is it equal to 1400000000?", population == 1400000000)
print("   card number   ->", card_number)

## Division: the one that catches everyone

`/` and `//` answer different questions, and `//` rounds **towards negative
infinity** rather than towards zero.

In [ ]:
# True division ALWAYS returns a float, even when it divides evenly.
print("True division with /:")
print("   7 / 2   =", 7 / 2, "  type:", type(7 / 2).__name__)
print("   10 / 5  =", 10 / 5, "  type:", type(10 / 5).__name__, "<- not 2")
print("   10 / 3  =", 10 / 3)

# Floor division returns an int when both operands are ints.
print("")
print("Floor division with //:")
print("   7 // 2  =", 7 // 2, "   type:", type(7 // 2).__name__)
print("   10 // 5 =", 10 // 5)
print("   10 // 3 =", 10 // 3)

# The modulo operator gives the remainder.
print("")
print("Remainder with %:")
print("   7 % 2   =", 7 % 2)
print("   10 % 3  =", 10 % 3)

# divmod does both at once.
print("")
print("divmod(17, 5) returns both:", divmod(17, 5))

In [ ]:
# The negative-number surprise: // rounds DOWN, not towards zero.
print("Floor division rounds towards negative infinity:")
print("")
print("   Expression    Python   'Truncation' would give")
print("   " + "-" * 48)

cases = [(7, 2), (-7, 2), (7, -2), (-7, -2)]

for numerator, denominator in cases:
    python_result = numerator // denominator
    # int() truncates towards zero, which is what C does.
    truncated = int(numerator / denominator)
    expression = f"{numerator} // {denominator}"
    print(f"   {expression:<13} {python_result:<8} {truncated}")

print("")
print("-7 // 2 is -4, not -3, because -4 is LOWER than -3.5.")
print("This is consistent with how % behaves in Python.")

# The relationship that always holds.
print("")
print("The invariant: (a // b) * b + (a % b) == a")
for numerator, denominator in cases:
    reconstructed = (numerator // denominator) * denominator + (numerator % denominator)
    print(f"   {numerator:>3} // {denominator:>2}: {reconstructed} == {numerator}  "
          f"{reconstructed == numerator}")

In [ ]:
# Python's % always takes the sign of the DIVISOR.
# In C and Java it takes the sign of the dividend - a real portability trap.

print("Python's modulo takes the divisor's sign:")
print("")
for numerator, denominator in [(7, 3), (-7, 3), (7, -3), (-7, -3)]:
    print(f"   {numerator:>3} % {denominator:>3} = {numerator % denominator:>3}")

print("")
print("This makes % genuinely useful for cyclic values:")

# Wrapping around a clock face works correctly even going backwards.
for hours in [-3, -1, 0, 13, 25]:
    print(f"   hour {hours:>3} on a 12-hour clock -> {hours % 12}")

print("")
print("In C, -3 % 12 would be -3, breaking the wrap.")

## Integer methods and useful built-ins

In [ ]:
value = 255

# bit_length tells you how many bits are needed.
print("Methods on int:")
print("   (255).bit_length()  ->", value.bit_length(), "bits")
print("   (255).bit_count()   ->", value.bit_count(), "ones in binary")
print("   (255).to_bytes(2)   ->", value.to_bytes(2, "big"))
print("   int.from_bytes(...) ->", int.from_bytes(b"\x00\xff", "big"))

# as_integer_ratio works on ints too, for consistency with floats.
print("   (255).as_integer_ratio() ->", value.as_integer_ratio())

print("")
print("Useful built-ins:")
print("   abs(-42)        ->", abs(-42))
print("   pow(2, 10)      ->", pow(2, 10))
print("   pow(2, 10, 100) ->", pow(2, 10, 100), "<- modular, used in cryptography")
print("   round(12345, -2)->", round(12345, -2), "<- negative digits round left")
print("   divmod(17, 5)   ->", divmod(17, 5))
print("   max/min/sum     ->", max(3, 7), min(3, 7), sum([1, 2, 3]))

## Performance: big integers are not free

Arbitrary precision costs speed. For everyday numbers this never matters; for
very large ones it does.

In [ ]:
import time

def time_multiplication(bits, repetitions=2000):
    """Time repeated multiplication of numbers of a given bit size."""
    left = 2 ** bits - 1
    right = 2 ** bits - 3

    start = time.perf_counter()
    for _ in range(repetitions):
        left * right
    return (time.perf_counter() - start) * 1000


print("Multiplying integers of increasing size (2000 times each):")
print("")
print("   Bits      Milliseconds")
print("   " + "-" * 26)

for bits in [32, 256, 2048, 8192]:
    elapsed = time_multiplication(bits)
    print(f"   {bits:<9} {elapsed:.2f}")

print("")
print("Small integers are fast. Very large ones are noticeably slower,")
print("because each operation processes more internal digits.")
print("")
print("Practical impact: almost none. You would need numbers with")
print("thousands of digits before this matters.")

In [ ]:
import sys

# Python 3.11+ limits how many digits an int can be CONVERTED to a string,
# to prevent denial-of-service from quadratic conversion algorithms.
if hasattr(sys, "get_int_max_str_digits"):
    limit = sys.get_int_max_str_digits()
    print("Maximum digits for int-to-string conversion:", limit)

    # The arithmetic is unlimited; only the string conversion is capped.
    huge = 10 ** (limit + 10)
    print("Computing a number beyond the limit: fine")
    print("   bit_length():", huge.bit_length())

    try:
        str(huge)
    except ValueError as error:
        print("")
        print("But converting it to a string:")
        print("   ", str(error)[:90])

    print("")
    print("Raise it with sys.set_int_max_str_digits() if you genuinely")
    print("need to print such numbers.")
else:
    print("This Python does not have the int-to-string digit limit.")

## Takeaways

1. Python integers have **no maximum** — arbitrary precision, no overflow, ever.
2. They are objects storing variable-length digits, so `sys.getsizeof()` grows
   with magnitude.
3. Four literal forms: decimal, `0b`, `0o`, `0x`. Underscores are ignored and
   purely for readability.
4. `/` is **true division** and always returns a `float` — `10 / 5` is `2.0`.
5. `//` is **floor division**, rounding **towards negative infinity**, so
   `-7 // 2` is `-4`.
6. `%` takes the sign of the **divisor**, which makes it correct for cyclic
   values like clock arithmetic.
7. `(a // b) * b + (a % b) == a` always holds.
8. Big-integer arithmetic is slower, and Python 3.11+ caps **string conversion**
   (not arithmetic) at 4300 digits by default.

## Try it yourself

1. Compute `2 ** 10000`. How many digits? How many bytes?
2. Work out `-7 // 2` and `-7 % 2` on paper, then check. Does the invariant hold?
3. Write the number 1000 in all four bases and confirm they are equal.
4. Use `%` to wrap a negative index around a list length. Why does it work?
5. Time `2 ** 100000` versus `2 ** 100`. Is the difference noticeable?